In [5]:
import pandas as pd
import sqlite3

# Load a CSV file using Pandas and display basic info.
df = pd.read_excel(r"D:\data-analyst-internship\data-analyst-internship\data-analyst-internship\week2\copy_SQL_Sales_Dataset_200_Rows.xlsx")

# convert order_date to string (YYYY-MM-DD) for clean SQL date filtering/sorting
df_sql = df.copy()
df_sql["order_date"] = pd.to_datetime(df_sql["order_date"]).dt.strftime(
    "%Y-%m-%d"
)

# Creating an in-memory SQLite database and writing df_sql to a table named 'sales'
conn = sqlite3.connect(":memory:")
df_sql.to_sql("sales", conn, index=False, if_exists="replace")

# Helper function to run SQL and display a Pandas DataFrame
def run_sql(query):
  return pd.read_sql_query(query, conn)



In [18]:
# Queries
query1 = "SELECT * FROM sales LIMIT 10;"
run_sql(query1)

,order_id,customer_name,order_date,category,sub_category,product_name,quantity,unit_price,total_price,region
0,1000,Caleb Davis,2025-01-31,Clothing,Jeans,Jeans Try,2,1197,2394,West
1,1001,Misty Herrera,2025-05-05,Clothing,Jeans,Jeans New,9,1509,13581,South
2,1002,Rick Soto,2025-06-09,Clothing,Shoes,Shoes Us,5,1046,5230,East
3,1003,Sean Haas,2025-02-27,Furniture,Desk,Desk Section,10,292,2920,South
4,1004,Ann Hall,2025-06-01,Furniture,Cabinet,Cabinet Center,2,548,1096,North
5,1005,Michelle White,2025-02-07,Furniture,Desk,Desk Point,2,2729,5458,South
6,1006,Breanna Gonzalez,2025-04-09,Grocery,Bread,Bread Black,6,4356,26136,East
7,1007,Angela Rivera,2025-06-18,Furniture,Chair,Chair Believe,7,2971,20797,West
8,1008,Emily Anderson,2025-02-28,Electronics,Mobile,Mobile Evening,10,701,7010,North
9,1009,Laura Anderson,2025-02-03,Grocery,Snacks,Snacks Capital,1,4454,4454,East


In [19]:
query2 = "SELECT order_id, customer_name, order_date, product_name, quantity, unit_price, region FROM sales WHERE region='West' LIMIT 10;"
run_sql(query2)

,order_id,customer_name,order_date,product_name,quantity,unit_price,region
0,1000,Caleb Davis,2025-01-31,Jeans Try,2,1197,West
1,1007,Angela Rivera,2025-06-18,Chair Believe,7,2971,West
2,1011,Valerie Hooper,2025-03-16,Couch Charge,10,1759,West
3,1016,Rachel Wilson,2025-04-04,Bread Skin,2,2413,West
4,1019,Debbie Turner,2025-03-22,Snacks Some,10,4499,West
5,1028,Abigail Powell,2025-05-23,Shoes Hear,1,2195,West
6,1035,Bradley Wilson,2025-06-20,Bread Something,9,1006,West
7,1036,John Hughes,2025-01-23,Couch Four,5,2671,West
8,1037,Eric Anderson,2025-01-12,Cabinet Him,1,3500,West
9,1039,Robert Bennett,2025-02-16,Desk Show,5,2697,West


In [16]:
query3 = """
SELECT
    customer_name,
    COUNT(order_id) AS total_orders,
    SUM(quantity) AS total_units,
    SUM(total_price) AS total_spent
FROM sales
GROUP BY customer_name
ORDER BY total_spent DESC
LIMIT 10;
"""
run_sql(query3)

,customer_name,total_orders,total_units,total_spent
0,Lynn Garrison,1,10,47940
1,Debbie Turner,1,10,44990
2,Megan Charles,1,10,44620
3,Michelle Beltran,1,9,42471
4,Rick Sanford,1,9,41211
5,Patrick Watson,1,8,37680
6,Christopher Meyer,1,8,37320
7,Todd Barnes,1,8,36152
8,Stephanie Martinez,1,8,34576
9,Kimberly Alvarado,1,7,31752


In [20]:
query4 = """
SELECT
    category,
    COUNT(order_id) AS order_count,
    SUM(total_price) AS total_revenue,
    ROUND(AVG(total_price), 2) AS avg_order_value
FROM sales
GROUP BY category
ORDER BY total_revenue DESC;
"""
run_sql(query4)

,category,order_count,total_revenue,avg_order_value
0,Furniture,59,714399,12108.46
1,Grocery,49,672147,13717.29
2,Electronics,52,549302,10563.50
3,Clothing,40,484259,12106.48


In [21]:
query5 = """
SELECT
    order_id,
    customer_name,
    category,
    region,
    total_price,
    CASE
        WHEN total_price >= 25000 THEN 'High Value'
        WHEN total_price >= 10000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS order_tier
FROM sales
ORDER BY total_price DESC
LIMIT 15;
"""
run_sql(query5)

,order_id,customer_name,category,region,total_price,order_tier
0,1125,Lynn Garrison,Grocery,East,47940,High Value
1,1019,Debbie Turner,Grocery,West,44990,High Value
2,1168,Megan Charles,Clothing,West,44620,High Value
3,1160,Michelle Beltran,Furniture,East,42471,High Value
4,1120,Rick Sanford,Furniture,South,41211,High Value
5,1070,Patrick Watson,Furniture,West,37680,High Value
6,1190,Christopher Meyer,Grocery,East,37320,High Value
7,1149,Todd Barnes,Furniture,North,36152,High Value
8,1114,Stephanie Martinez,Grocery,West,34576,High Value
9,1174,Kimberly Alvarado,Furniture,North,31752,High Value


In [22]:
query6 = """
SELECT
    order_id,
    customer_name,
    category,
    region,
    total_price
FROM sales
WHERE total_price > (
    SELECT AVG(total_price) FROM sales
)
ORDER BY total_price DESC;
"""
run_sql(query6)

,order_id,customer_name,category,region,total_price
0,1125,Lynn Garrison,Grocery,East,47940
1,1019,Debbie Turner,Grocery,West,44990
2,1168,Megan Charles,Clothing,West,44620
3,1160,Michelle Beltran,Furniture,East,42471
4,1120,Rick Sanford,Furniture,South,41211
...,...,...,...,...,...
76,1020,Roy Stewart,Electronics,North,13300
77,1049,Melissa Ruiz,Clothing,South,13300
78,1145,Sheila Booker,Electronics,North,12817
79,1189,Laura Smith,Clothing,North,12222


In [ ]:
conn.execute("DROP TABLE IF EXISTS regional_targets;")
conn.execute("""
CREATE TABLE regional_targets (
    region TEXT PRIMARY KEY,
    quarterly_target REAL
);
""")
conn.executemany(
    "INSERT INTO regional_targets (region, quarterly_target) VALUES (?, ?);",
    [
        ("North", 500000),
        ("South", 600000),
        ("East", 650000),
        ("West", 550000),
    ],
)
conn.commit()

query = """
SELECT
    s.region,
    COUNT(s.order_id) AS total_orders,
    SUM(s.total_price) AS actual_revenue,
    t.quarterly_target,
    ROUND(SUM(s.total_price) - t.quarterly_target, 2) AS variance_from_target
FROM sales s
LEFT JOIN regional_targets t
    ON s.region = t.region
GROUP BY s.region, t.quarterly_target
ORDER BY actual_revenue DESC;
"""
run_sql(query)